In [4]:
import pulp as pl
import os
import pandas as pd
import numpy as np
import kagglehub

In [ ]:

# # Download latest version
# path = kagglehub.dataset_download("himelsarder/cinema-hall-ticket-sales-and-customer-behavior")

# print("Path to dataset files:", path)

100%|██████████| 14.9k/14.9k [00:00<00:00, 8.99MB/s]

Extracting files...
Path to dataset files: /home/codespace/.cache/kagglehub/datasets/himelsarder/cinema-hall-ticket-sales-and-customer-behavior/versions/1


In [ ]:
# # List files in the dataset directory
# files = os.listdir(path)
# print("Files in dataset:", files)

# # Load the main dataset (adjust filename as needed)
# df = pd.read_csv(os.path.join(path, files[0]))
# print(df.head())

Files in dataset: ['cinema_hall_ticket_sales.csv']
  Ticket_ID  Age  Ticket_Price Movie_Genre Seat_Type Number_of_Person  \
0     N4369   55         12.27      Comedy  Standard                7   
1     B8091   35         19.02       Drama  Standard            Alone   
2     V6341   55         22.52      Horror       VIP                3   
3     B3243   53         23.01       Drama  Standard                6   
4     I3814   30         21.81      Comedy       VIP                4   

  Purchase_Again  
0             No  
1            Yes  
2             No  
3            Yes  
4            Yes  


In [52]:
# Create a sample DataFrame to simulate the cinema hall ticket sales data
data = {
    'Movie': ['Action Hero', 'Romantic Sunset', 'Sci-Fi Odyssey', 'Comedy Club', 'Horror Night'],
    'Showtimes_Per_Day': [4, 3, 5, 4, 2],
    'Ticket_Price': [12.00, 10.00, 14.00, 9.00, 11.00],
    'Estimated_Demand': [200, 150, 220, 180, 130],
    'Marketing_Spend': [5000, 3000, 7000, 2000, 4000],
    'Running_Time_Minutes': [120, 110, 140, 95, 105],
    'Genre': ['Action', 'Romance', 'Sci-Fi', 'Comedy', 'Horror'],
    'Screen_Availability': [2, 1, 2, 1, 1]
}

df_movies = pd.DataFrame(data)
print(df_movies)


             Movie  Showtimes_Per_Day  Ticket_Price  Estimated_Demand  \
0      Action Hero                  4          12.0               200   
1  Romantic Sunset                  3          10.0               150   
2   Sci-Fi Odyssey                  5          14.0               220   
3      Comedy Club                  4           9.0               180   
4     Horror Night                  2          11.0               130   

   Marketing_Spend  Running_Time_Minutes    Genre  Screen_Availability  
0             5000                   120   Action                    2  
1             3000                   110  Romance                    1  
2             7000                   140   Sci-Fi                    2  
3             2000                    95   Comedy                    1  
4             4000                   105   Horror                    1  


In [53]:
#define the problem
problem = pl.LpProblem("Cinema_Hall_Optimization", pl.LpMaximize)

In [55]:

# Define the decision variables
showtimes = pl.LpVariable.dicts("Showtimes", df_movies['Movie'], lowBound=0, upBound=None, cat='Integer')
tickets_sold = pl.LpVariable.dicts("Tickets_Sold", df_movies['Movie'], lowBound=0, cat='Integer')
ticket_price = dict(zip(df_movies['Movie'], df_movies['Ticket_Price']))  # Assuming df_movies has a 'Price' column

# Define the objective function
problem += pl.lpSum([
    tickets_sold[movie] * ticket_price[movie] - showtimes[movie] 
    for movie in df_movies['Movie']
]), "Net_Revenue_With_Showtimes"

# Define the constraints
# 1. Showtimes and tickets sold constraints
for movie in df_movies['Movie'].unique():
    max_demand = df_movies.loc[df_movies['Movie'] == movie, 'Estimated_Demand'].values[0] * df_movies.loc[df_movies['Movie'] == movie, 'Showtimes_Per_Day'].values[0]
    problem += tickets_sold[movie] <= max_demand, f"Max_Tickets_Sold_{movie}"
    problem += tickets_sold[movie] >= 0, f"Min_Tickets_Sold_{movie}"
    problem += showtimes[movie] >= 1, f"Min_Showtimes_{movie}"

# 2. Total screen availability constraint
problem += pl.lpSum([showtimes[movie] for movie in df_movies['Movie']]) <= df_movies['Screen_Availability'].sum(), "Total_Screen_Availability"

# 3. Total running time constraint (example: total running time per day <= 1000 minutes)
total_running_time_limit = 1440  # Example value in minutes (e.g., 24 hours * 60 minutes)
problem += pl.lpSum([showtimes[movie] * df_movies.loc[df_movies['Movie'] == movie, 'Running_Time_Minutes'].values[0] for movie in df_movies['Movie']]) <= total_running_time_limit, "Total_Running_Time_Limit"

# 4. Total marketing spend constraint (example: total marketing spend <= budget)
budget = df_movies['Marketing_Spend'].sum()
problem += pl.lpSum([df_movies.loc[df_movies['Movie'] == movie, 'Marketing_Spend'].values[0] for movie in df_movies['Movie']]) <= budget, "Total_Marketing_Spend"

# 5. Number of seats per screen constraint (example: each movie can have a maximum of 250 seats sold per showtime)
max_seats_per_showtime = 250  # Example value
for movie in df_movies['Movie']:
    problem += tickets_sold[movie] <= showtimes[movie] * max_seats_per_showtime, f"Max_Seats_Sold_{movie}"

# 6. Showtimes per day constraint (example: each movie must have atleast 1 showtime per day, but no maximum)
for movie in df_movies['Movie']:
    problem += showtimes[movie] <= df_movies['Showtimes_Per_Day'].sum(), f"Max_Showtimes_{movie}"

# Solve the problem
problem.solve()

# Print the results
print("Status:", pl.LpStatus[problem.status])
print("Total Revenue:", pl.value(problem.objective))
for movie in df_movies['Movie']:
    print(f"Showtimes for {movie}: {showtimes[movie].varValue}")
    print(f"Tickets Sold for {movie}: {tickets_sold[movie].varValue}")

# Display the results in a DataFrame
results_df = pd.DataFrame({
    "Movie": df_movies['Movie'],
    "Showtimes": [showtimes[movie].varValue for movie in df_movies['Movie']],
    "Tickets Sold": [tickets_sold[movie].varValue for movie in df_movies['Movie']],
    "Revenue": [showtimes[movie].varValue * ticket_price[movie] for movie in df_movies['Movie']]
})

Status: Optimal
Total Revenue: 20993.0
Showtimes for Action Hero: 1.0
Tickets Sold for Action Hero: 250.0
Showtimes for Romantic Sunset: 1.0
Tickets Sold for Romantic Sunset: 250.0
Showtimes for Sci-Fi Odyssey: 3.0
Tickets Sold for Sci-Fi Odyssey: 750.0
Showtimes for Comedy Club: 1.0
Tickets Sold for Comedy Club: 250.0
Showtimes for Horror Night: 1.0
Tickets Sold for Horror Night: 250.0


In [ ]:
results_df